# Notebook 10 – Feature Selection

This notebook builds a feature set from `customer_transactions_raw.csv` and applies several feature selection techniques to decide which features are worth keeping for a supervised task: predicting whether a customer is a **high-value customer** (purchase amount above the median).

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('customer_transactions_raw.csv')
df['age_numeric'] = pd.to_numeric(df['age'], errors='coerce')
df['purchase_amount_clean'] = df['purchase_amount'].astype(str).str.replace('$', '', regex=False)
df['purchase_amount_clean'] = pd.to_numeric(df['purchase_amount_clean'], errors='coerce')
df['gender_clean'] = df['gender'].str.strip().str.lower().replace({'m': 'male', 'f': 'female'})
df['membership_ordinal'] = df['membership_type'].str.strip().str.lower().map({'bronze': 1, 'silver': 2, 'gold': 3, 'platinum': 4})
df.shape

(1000, 16)

## 1. What is Feature Selection?

Feature selection is the process of choosing which input columns (features) a model should actually use, rather than feeding it every available column. It is distinct from feature *engineering* (creating new features) — selection is about deciding which existing features earn their place in the model.

## 2. Why Feature Selection?

- **Simpler, faster models** — fewer features means less computation and memory.
- **Reduced overfitting** — fewer irrelevant inputs means less chance the model memorizes noise.
- **Better interpretability** — a smaller, meaningful feature set is easier to explain to stakeholders.
- **Improved accuracy** — removing noisy or redundant features can directly improve generalization, since noisy features add variance without adding signal.

## 3. Relevant Features

Relevant features carry real information about the target we're predicting. In this dataset, we'd expect `annual_income`, `membership_ordinal`, and `quantity` to be meaningfully related to `purchase_amount_clean` (our proxy for customer value), since higher income and higher membership tier plausibly relate to spending more.

In [3]:
target = (df['purchase_amount_clean'] > df['purchase_amount_clean'].median()).astype(int)
target.value_counts()

purchase_amount_clean
0    500
1    500
Name: count, dtype: int64

## 4. Irrelevant Features

Irrelevant features carry no real signal about the target. Two clear examples exist directly in this dataset:

- **`notes`** — this column is 100% missing (see below), so it cannot possibly inform any prediction.
- **`customer_id`** — an arbitrary identifier assigned at signup; it has no causal or statistical relationship with spending behavior, even though it's a number the model *could* technically use.

In [4]:
df['notes'].notna().sum(), df['notes'].isna().mean()

(np.int64(0), np.float64(1.0))

In [5]:
df['customer_id'].nunique(), len(df)

(950, 1000)

## 5. Redundant Features

Redundant features duplicate information already captured by another feature. If two columns are almost perfectly correlated, keeping both adds no new signal — only extra noise and, for linear models, multicollinearity risk.

**Example we construct here:** an `annual_income_thousands` column, which is just `annual_income` divided by 1000. It's mathematically identical information in a different unit, so it is fully redundant with `annual_income`.

In [6]:
df['annual_income_thousands'] = df['annual_income'] / 1000
df[['annual_income', 'annual_income_thousands']].corr()

,annual_income,annual_income_thousands
annual_income,1.0,1.0
annual_income_thousands,1.0,1.0


## 6. Correlation-Based Selection

For numeric features, computing correlation with the target (and between features) helps flag both relevant and redundant columns:

- **High correlation with target** → potentially relevant.
- **High correlation between two features** → potentially redundant; consider dropping one.

In [8]:
numeric_features = df[['age_numeric', 'annual_income', 'annual_income_thousands', 'purchase_amount_clean', 'quantity', 'rating', 'membership_ordinal', 'customer_id']].copy()
numeric_features['target'] = target
correlation_with_target = numeric_features.corr()['target'].drop('target').sort_values(ascending=False)
correlation_with_target

purchase_amount_clean      0.108686
membership_ordinal         0.033606
customer_id                0.026213
rating                    -0.001357
annual_income_thousands   -0.018207
annual_income             -0.018207
quantity                  -0.033659
age_numeric               -0.061960
Name: target, dtype: float64

In [9]:
feature_correlation_matrix = numeric_features.drop(columns=['target']).corr()
feature_correlation_matrix

,age_numeric,annual_income,annual_income_thousands,purchase_amount_clean,quantity,rating,membership_ordinal,customer_id
age_numeric,1.000000,-0.024449,-0.024449,0.017445,0.041436,0.026343,-0.003148,0.011373
annual_income,-0.024449,1.000000,1.000000,-0.004380,-0.065635,-0.050712,0.030991,0.080049
annual_income_thousands,-0.024449,1.000000,1.000000,-0.004380,-0.065635,-0.050712,0.030991,0.080049
purchase_amount_clean,0.017445,-0.004380,-0.004380,1.000000,0.000575,-0.025372,0.029660,0.013055
quantity,0.041436,-0.065635,-0.065635,0.000575,1.000000,0.054190,0.003629,-0.029088
rating,0.026343,-0.050712,-0.050712,-0.025372,0.054190,1.000000,0.061157,-0.002908
membership_ordinal,-0.003148,0.030991,0.030991,0.029660,0.003629,0.061157,1.000000,-0.042154
customer_id,0.011373,0.080049,0.080049,0.013055,-0.029088,-0.002908,-0.042154,1.000000


**Reading the results:** `purchase_amount_clean` correlates with `target` almost by definition (target was derived from it, so it is excluded from the actual feature set below). `annual_income` and `annual_income_thousands` are perfectly correlated with each other (redundant pair — one should be dropped). `customer_id` shows negligible correlation with the target, consistent with it being an irrelevant identifier.

## 7. Variance Threshold

A feature with (near) zero variance carries no information at all, since it barely changes across rows. `notes`, being 100% missing, has undefined/zero variance and would be automatically flagged by a variance-threshold filter.

In [10]:
from sklearn.feature_selection import VarianceThreshold
candidate_numeric = df[['age_numeric', 'annual_income', 'quantity', 'rating', 'membership_ordinal']].fillna(df[['age_numeric', 'annual_income', 'quantity', 'rating', 'membership_ordinal']].median())
selector = VarianceThreshold(threshold=0.01)
selector.fit(candidate_numeric)
pd.Series(selector.variances_, index=candidate_numeric.columns).sort_values()

membership_ordinal    9.663360e-01
rating                2.171804e+00
quantity              6.657984e+00
age_numeric           2.003576e+02
annual_income         2.990603e+15
dtype: float64

In [11]:
notes_variance = df['notes'].var()
notes_variance

np.float64(nan)

## 8. Univariate Feature Selection

Univariate methods score each feature independently against the target (ignoring interactions between features), then keep the top-K or top-percentile scoring features. `SelectKBest` with an ANOVA F-test is a common choice for a numeric-features/classification-target setup like ours.

In [12]:
from sklearn.feature_selection import SelectKBest, f_classif
feature_matrix = df[['age_numeric', 'annual_income', 'quantity', 'rating', 'membership_ordinal']].fillna(
    df[['age_numeric', 'annual_income', 'quantity', 'rating', 'membership_ordinal']].median()
)
selector_kbest = SelectKBest(score_func=f_classif, k=3)
selector_kbest.fit(feature_matrix, target)
pd.Series(selector_kbest.scores_, index=feature_matrix.columns).sort_values(ascending=False)

age_numeric           3.689041
quantity              1.109858
membership_ordinal    0.930357
annual_income         0.330770
rating                0.001838
dtype: float64

## 9. Mutual Information

Mutual information measures how much knowing a feature's value reduces uncertainty about the target, and unlike correlation, it can capture **non-linear** relationships between a feature and the target.

In [13]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(feature_matrix, target, random_state=42)
pd.Series(mi_scores, index=feature_matrix.columns).sort_values(ascending=False)

age_numeric           0.020293
annual_income         0.011465
quantity              0.005306
rating                0.000000
membership_ordinal    0.000000
dtype: float64

## 10. Recursive Feature Elimination

Recursive Feature Elimination (RFE) fits a model repeatedly, each time removing the least important feature(s), until the desired number of features remains. Unlike univariate methods, RFE accounts for feature *interactions* since it evaluates features together within a real model.

In [14]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
rfe_model = LogisticRegression(max_iter=1000)
rfe_selector = RFE(estimator=rfe_model, n_features_to_select=3)
rfe_selector.fit(feature_matrix, target)
pd.DataFrame({
    'feature': feature_matrix.columns,
    'selected': rfe_selector.support_,
    'rank': rfe_selector.ranking_,
}).sort_values('rank')

,feature,selected,rank
0,age_numeric,True,1
1,annual_income,True,1
2,quantity,True,1
4,membership_ordinal,False,2
3,rating,False,3


## 11. Feature Importance

Tree-based ensemble models (like Random Forest) provide a built-in feature importance score based on how much each feature reduces impurity across all trees. This is a model-driven alternative to the statistical tests above, and naturally accounts for interactions and non-linearities.

In [15]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(feature_matrix, target)
pd.Series(rf_model.feature_importances_, index=feature_matrix.columns).sort_values(ascending=False)

annual_income         0.368217
age_numeric           0.290726
quantity              0.148298
rating                0.115981
membership_ordinal    0.076778
dtype: float64

## 12. Putting It Together

Combining the checks above for this dataset:

- **Drop:** `notes` (zero variance / entirely missing) and `customer_id` (irrelevant identifier, confirmed by near-zero correlation, low univariate score, and low importance).
- **Drop one of a redundant pair:** `annual_income_thousands` duplicates `annual_income` exactly — keep only `annual_income`.
- **Keep:** `annual_income`, `quantity`, `membership_ordinal`, and `rating` — these show meaningful signal across multiple methods (correlation, univariate F-test, mutual information, RFE, and tree-based importance).
- **Borderline:** `age_numeric` — showed weaker but non-zero signal across methods; whether to keep it depends on the modeling budget and whether simplicity or completeness is prioritized.

In [14]:
final_selected_features = ['annual_income', 'quantity', 'membership_ordinal', 'rating']
final_feature_set = df[final_selected_features].copy()
final_feature_set['target'] = target
final_feature_set.to_csv('customer_transactions_selected_features.csv', index=False)
final_feature_set.head()

,annual_income,quantity,membership_ordinal,rating,target
0,80242.98,1.0,3.0,1,0
1,NaN,1.0,2.0,4,1
2,56285.40,2.0,3.0,4,1
3,81878.74,9.0,2.0,4,0
4,100566.42,1.0,2.0,4,0


## 13. How Unnecessary Features Affect Machine Learning Models

- **Overfitting risk increases.** Irrelevant features (like `customer_id`) give the model extra ways to find spurious patterns in the training data that don't generalize to new data.
- **Curse of dimensionality.** As more features are added, the data needed to reliably estimate relationships grows exponentially; with a fixed dataset size, adding unhelpful columns makes every relationship harder to estimate precisely.
- **Slower training and inference.** More features mean more computation, which matters at scale even if each individual feature seems cheap to include.
- **Multicollinearity distorts linear models.** Redundant features (like `annual_income` and `annual_income_thousands`) can make coefficient estimates unstable and hard to interpret in linear/logistic regression, since the model can't tell which of two near-identical columns deserves the credit.
- **Noise dilutes signal.** In distance-based or tree-based models, irrelevant features can add noise that obscures genuinely useful splits or distance comparisons, especially with a limited number of trees or limited data.
- **Harder interpretability and maintenance.** A model with lots of unnecessary features is harder to explain to stakeholders and more fragile to maintain, since every extra column is one more thing that can silently break or drift in production data.
- **Zero-variance features waste computation for zero benefit.** A feature like `notes`, which is entirely missing, contributes nothing predictive while still occupying a column that must be stored, validated, and processed.